# Crear RAG — Normativa de IA y protección de datos

Pipeline para preparar una base documental de normativa de IA y protección de datos aplicable en España y la UE:

1. Carga y validación de documentos desde `data/`
2. Limpieza del texto conservando referencias legales
3. Troceado por artículo con límites y solapamiento medidos en tokens
4. Embeddings con un modelo de HuggingFace ejecutado en local
5. Construcción y persistencia de un índice FAISS con metadatos de cita
6. Evaluación de consultas de recuperación representativas

El corpus incluye normas de IA y de protección de datos; consulta los metadatos de cada fuente y verifica su versión en la publicación oficial antes de usar los resultados con fines jurídicos.

Este notebook es el único punto de creación de `rag_store/`. `rag.py`, `app.py` y `chat.py` solo consultan el índice generado; no lo construyen ni lo modifican.

## Antes de empezar

Coloca los documentos en formato Markdown (`.md`) en `data/`. Si hay varias copias o versiones de una misma norma, el cargador se detiene para evitar indexarlas juntas; selecciona explícitamente una fuente/version.

## 1. Dependencias

In [1]:
%pip install -q sentence-transformers faiss-cpu numpy

Note: you may need to restart the kernel to use updated packages.


## 2. Imports y configuración

In [3]:
import json
import re
import warnings
from pathlib import Path

import faiss
from sentence_transformers import SentenceTransformer

DATA_DIR = Path("data")
OUTPUT_DIR = Path("rag_store")
OUTPUT_DIR.mkdir(exist_ok=True)

EMBEDDING_MODEL = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
CHUNK_SIZE_TOKENS = 100
CHUNK_OVERLAP_TOKENS = 20
CORPUS_AS_OF = "10/11/2026"

## 3. Carga de documentos

In [4]:
DOCUMENT_METADATA = {
    "BOE-A-2018-16673-consolidado.md": {
        "title": "Ley Orgánica 3/2018, de Protección de Datos Personales y garantía de los derechos digitales",
        "legal_reference": "BOE-A-2018-16673",
    },
    "BOE-A-2021-8806-consolidado.md": {
        "title": "Ley Orgánica 7/2021, de protección de datos personales tratados para fines penales",
        "legal_reference": "BOE-A-2021-8806",
    },
    "BOE-A-2023-18911-consolidado.md": {
        "title": "Real Decreto 729/2023, Estatuto de la Agencia Española de Supervisión de Inteligencia Artificial",
        "legal_reference": "BOE-A-2023-18911",
    },
    "L00001-00088.md": {
        "title": "Reglamento (UE) 2016/679, Reglamento General de Protección de Datos",
        "legal_reference": "Reglamento (UE) 2016/679",
    },
    "Reglamento-Europeo-IA.md": {
        "title": "Reglamento (UE) 2024/1689, Reglamento de Inteligencia Artificial",
        "legal_reference": "Reglamento (UE) 2024/1689",
    },
}


def document_metadata(path, content):
    known = DOCUMENT_METADATA.get(path.name, {})
    boe_reference = re.search(r"\bBOE-A-\d{4}-\d+\b", content, re.IGNORECASE)
    eu_reference = re.search(r"Reglamento\s*\(UE\)\s*\d{4}/\d+", content, re.IGNORECASE)
    detected_reference = boe_reference.group(0) if boe_reference else (
        eu_reference.group(0) if eu_reference else path.stem
    )
    version_match = re.search(
        r"(?im)^#{1,6}\s*TEXTO CONSOLIDADO\s*(.*)$", content
    )
    source_version = (
        version_match.group(1).strip(" .:-")
        if version_match and version_match.group(1).strip(" .:-")
        else CORPUS_AS_OF
    )
    return {
        "title": known.get("title", path.stem),
        "legal_reference": known.get("legal_reference", detected_reference),
        "source_version": source_version,
    }


def load_documents(data_dir):
    if not data_dir.exists():
        raise FileNotFoundError(
            f"No existe la carpeta '{data_dir}'. Crea 'data/' y añade documentos Markdown (.md)."
        )

    documents = []
    for path in sorted(data_dir.rglob("*.md")):
        if not path.is_file():
            continue

        content = path.read_text(encoding="utf-8")

        if not content.strip():
            raise ValueError(f"No se pudo extraer texto de '{path}'. Revisa el documento.")
        if len(content.strip()) < 500:
            warnings.warn(
                f"El documento '{path}' solo contiene {len(content.strip())} caracteres; "
                "comprueba que la extracción sea completa.",
                stacklevel=2,
            )

        relative_source = f"{data_dir.name}/{path.relative_to(data_dir).as_posix()}"
        documents.append({
            "source": relative_source,
            "content": content,
            **document_metadata(path, content),
        })

    if not documents:
        raise ValueError(f"No se encontraron documentos Markdown (.md) en '{data_dir}'.")

    by_reference = {}
    for doc in documents:
        by_reference.setdefault(doc["legal_reference"], []).append(doc)

    duplicates = {
        reference: [doc["source"] for doc in copies]
        for reference, copies in by_reference.items()
        if len(copies) > 1
    }
    if duplicates:
        raise ValueError(
            f"Hay varias fuentes/versiones para las mismas normas: {duplicates}. "
            "Selecciona una fuente explícitamente para evitar indexar duplicados o mezclar versiones."
        )

    return documents


documents = load_documents(DATA_DIR)
print(f"Documentos cargados: {len(documents)}")
for doc in documents:
    print(
        f"- {doc['title']} | {doc['legal_reference']} | {doc['source']} "
        f"({len(doc['content'])} caracteres) | versión: {doc['source_version']}"
    )

Documentos cargados: 5
- Ley Orgánica 3/2018, de Protección de Datos Personales y garantía de los derechos digitales | BOE-A-2018-16673 | data/BOE-A-2018-16673-consolidado.md (281484 caracteres) | versión: Última modificación: 27 de diciembre de 2025
- Ley Orgánica 7/2021, de protección de datos personales tratados para fines penales | BOE-A-2021-8806 | data/BOE-A-2021-8806-consolidado.md (206050 caracteres) | versión: Última modificación: 29 de julio de 2022
- Real Decreto 729/2023, Estatuto de la Agencia Española de Supervisión de Inteligencia Artificial | BOE-A-2023-18911 | data/BOE-A-2023-18911-consolidado.md (115651 caracteres) | versión: Última modificación: sin modificaciones
- Reglamento (UE) 2016/679, Reglamento General de Protección de Datos | Reglamento (UE) 2016/679 | data/L00001-00088.md (384478 caracteres) | versión: 10/11/2026
- Reglamento (UE) 2024/1689, Reglamento de Inteligencia Artificial | Reglamento (UE) 2024/1689 | data/Reglamento-Europeo-IA.md (673937 caracteres)

## 4. Limpieza del texto

Se eliminan guiones de corte, espacios repetidos y saltos de línea excesivos. La estructura de artículos se conserva para el troceado posterior.

In [5]:
def clean_text(text):
    text = text.replace("\u00ad", "")
    text = re.sub(r"(\w)-\n(\w)", r"\1\2", text)
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()

## 5. Troceado por secciones legales

Se cargan el modelo y su tokenizador para limitar cada chunk según tokens, no caracteres. Los límites de artículos y disposiciones se respetan y cada chunk conserva metadatos para citar la fuente.

In [6]:
model = SentenceTransformer(EMBEDDING_MODEL)
tokenizer = model.tokenizer
special_tokens = tokenizer.num_special_tokens_to_add(pair=False)
MAX_CHUNK_TOKENS = min(CHUNK_SIZE_TOKENS, model.max_seq_length - special_tokens)

if MAX_CHUNK_TOKENS <= 0 or CHUNK_OVERLAP_TOKENS >= MAX_CHUNK_TOKENS:
    raise ValueError("El tamaño del chunk y el solapamiento no son válidos para el modelo.")

LEGAL_SECTION_HEADING = re.compile(
    r"^\s*(?:#{1,6}\s*)?((?:Artículo(?:\s+único)?|Disposición\s+"
    r"(?:adicional|transitoria|derogatoria|final)|Anexo)\b.*)$",
    re.IGNORECASE,
)


def split_legal_sections(text):
    sections = []
    heading = "Preámbulo y disposiciones iniciales"
    lines = []

    def save_section():
        section_text = "\n".join(lines).strip()
        if section_text:
            article_match = re.search(
                r"\bArtículo\s+(único|\d+[A-Za-z]?)\b", heading, re.IGNORECASE
            )
            sections.append({
                "heading": heading,
                "article": article_match.group(1) if article_match else None,
                "text": section_text,
            })

    for line in text.splitlines():
        match = LEGAL_SECTION_HEADING.match(line)
        if match:
            save_section()
            heading = match.group(1).strip()
            lines = [line]
        else:
            lines.append(line)
    save_section()
    return sections


def token_count(text):
    return len(tokenizer.encode(text, add_special_tokens=False))


def split_oversized_sentence(sentence, max_tokens):
    pieces = []
    current_words = []
    for word in sentence.split():
        candidate = " ".join(current_words + [word])
        if token_count(candidate) <= max_tokens:
            current_words.append(word)
            continue
        if not current_words:
            raise ValueError("Una palabra excede el límite de tokens configurado.")
        pieces.append(" ".join(current_words))
        current_words = [word]
        if token_count(word) > max_tokens:
            raise ValueError("Una palabra excede el límite de tokens configurado.")
    if current_words:
        pieces.append(" ".join(current_words))
    return pieces


def chunk_text(text, max_tokens=MAX_CHUNK_TOKENS, overlap_tokens=CHUNK_OVERLAP_TOKENS):
    text = re.sub(r"\s+", " ", text).strip()
    if not text:
        return []

    sentences = re.split(r"(?<=[.!?])\s+", text)
    chunks = []
    current_sentences = []

    for sentence in sentences:
        if not sentence:
            continue
        pieces = split_oversized_sentence(sentence, max_tokens)
        for piece in pieces:
            if not current_sentences:
                current_sentences = [piece]
                continue

            candidate = " ".join(current_sentences + [piece])
            if token_count(candidate) <= max_tokens:
                current_sentences.append(piece)
                continue

            chunks.append(" ".join(current_sentences))
            overlap = []
            for previous in reversed(current_sentences):
                candidate_overlap = [previous] + overlap
                if not overlap_tokens or token_count(" ".join(candidate_overlap)) > overlap_tokens:
                    break
                overlap = candidate_overlap
            while overlap and token_count(" ".join(overlap + [piece])) > max_tokens:
                overlap.pop(0)
            current_sentences = overlap + [piece]

    if current_sentences:
        chunks.append(" ".join(current_sentences))

    if any(token_count(chunk) > max_tokens for chunk in chunks):
        raise ValueError("Se generó un chunk que supera el límite de tokens.")
    return chunks


chunks = []
for doc in documents:
    cleaned = clean_text(doc["content"])
    chunk_id = 0
    for section in split_legal_sections(cleaned):
        for chunk in chunk_text(section["text"]):
            chunks.append({
                "source": doc["source"],
                "title": doc["title"],
                "legal_reference": doc["legal_reference"],
                "source_version": doc["source_version"],
                "section": section["heading"],
                "article": section["article"],
                "chunk_id": chunk_id,
                "text": chunk,
            })
            chunk_id += 1

print(f"Chunks generados: {len(chunks)}")
print(f"Límite efectivo: {MAX_CHUNK_TOKENS} tokens; solapamiento objetivo: {CHUNK_OVERLAP_TOKENS}")
if not chunks:
    raise ValueError("No se han generado chunks. Revisa la extracción de texto de las fuentes.")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 44148.23it/s]
[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (136 > 128). Running this sequence through the model will result in indexing errors


Chunks generados: 4820
Límite efectivo: 100 tokens; solapamiento objetivo: 20


## 6. Embeddings en local

Se usa `sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2`, un modelo multilingüe para español. El límite de chunk se deriva de `model.max_seq_length` para reducir truncamientos.

In [7]:
texts = [chunk["text"] for chunk in chunks]
embeddings = model.encode(
    texts,
    batch_size=32,
    show_progress_bar=True,
    normalize_embeddings=True,
    convert_to_numpy=True,
).astype("float32")

print(f"Dimensiones de embeddings: {embeddings.shape}")

Batches: 100%|██████████| 151/151 [00:16<00:00,  9.27it/s]


Dimensiones de embeddings: (4820, 384)


## 7. Construcción y persistencia del índice FAISS

Se usa `IndexFlatIP` (producto interno) junto con embeddings normalizados, lo que equivale a similitud coseno. El índice se acompaña de chunks citables y un manifiesto de fuentes/versiones.

In [8]:
index = faiss.IndexFlatIP(embeddings.shape[1])
index.add(embeddings)

index_path = OUTPUT_DIR / "index.faiss"
chunks_path = OUTPUT_DIR / "chunks.json"
meta_path = OUTPUT_DIR / "metadata.json"

faiss.write_index(index, str(index_path))
chunks_path.write_text(
    json.dumps(chunks, ensure_ascii=False, indent=2), encoding="utf-8"
)
meta_path.write_text(
    json.dumps(
        {
            "embedding_model": EMBEDDING_MODEL,
            "embedding_max_seq_length": model.max_seq_length,
            "num_documents": len(documents),
            "num_chunks": len(chunks),
            "chunk_size_tokens": MAX_CHUNK_TOKENS,
            "chunk_overlap_tokens": CHUNK_OVERLAP_TOKENS,
            "dimension": int(embeddings.shape[1]),
            "corpus_as_of": CORPUS_AS_OF,
            "documents": [
                {
                    "source": doc["source"],
                    "title": doc["title"],
                    "legal_reference": doc["legal_reference"],
                    "source_version": doc["source_version"],
                }
                for doc in documents
            ],
        },
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)

print(f"Índice guardado en: {index_path}")
print(f"Chunks guardados en: {chunks_path}")
print(f"Metadatos guardados en: {meta_path}")

Índice guardado en: rag_store/index.faiss
Chunks guardados en: rag_store/chunks.json
Metadatos guardados en: rag_store/metadata.json


## 8. Búsqueda y evaluación de recuperación

Los resultados incluyen la norma, el artículo y la fuente para poder rastrear la respuesta. Las consultas de abajo sirven como comprobaciones iniciales con resultados esperados; revisa manualmente los chunks recuperados antes de considerar satisfactoria la recuperación.

In [9]:
def search(query, k=5):
    if not isinstance(query, str) or not query.strip():
        raise ValueError("La consulta no puede estar vacía.")
    if k < 1:
        raise ValueError("k debe ser al menos 1.")

    query_embedding = model.encode(
        [query],
        normalize_embeddings=True,
        convert_to_numpy=True,
    ).astype("float32")
    scores, indices = index.search(query_embedding, min(k, index.ntotal))
    return [
        (float(score), chunks[chunk_index])
        for score, chunk_index in zip(scores[0], indices[0])
        if chunk_index != -1
    ]


EVALUATION_CASES = [
    {
        "query": "¿Qué medidas de gestión de riesgos se exigen para los sistemas de IA de alto riesgo?",
        "legal_reference": "Reglamento (UE) 2024/1689",
        "article": "9",
    },
    {
        "query": "¿Qué establece el derecho de acceso de los interesados a sus datos personales?",
        "legal_reference": "Reglamento (UE) 2016/679",
        "article": "15",
    },
    {
        "query": "¿Cómo se regula el derecho de acceso a los datos personales en España?",
        "legal_reference": "BOE-A-2018-16673",
        "article": "13",
    },
]

for case in EVALUATION_CASES:
    results = search(case["query"], k=5)
    expected_found = any(
        result["legal_reference"] == case["legal_reference"]
        and result["article"] == case["article"]
        for _, result in results
    )
    status = "ESPERADO EN TOP 5" if expected_found else "NO ENCONTRADO EN TOP 5"
    print(
        f"{status}: {case['legal_reference']}, artículo {case['article']} "
        f"— {case['query']}"
    )
    for score, chunk in results[:3]:
        citation = chunk["article"] or chunk["section"]
        print(f"  [{score:.3f}] {chunk['legal_reference']} — {citation} — {chunk['source']}")

print("\nPrueba manual adicional:")
for score, chunk in search("¿Qué obligaciones establece la normativa de IA en España?", k=5):
    citation = chunk["article"] or chunk["section"]
    print(f"[{score:.3f}] {chunk['legal_reference']} — {citation} — {chunk['source']}")
    print(chunk["text"][:400])
    print("-" * 80)

ESPERADO EN TOP 5: Reglamento (UE) 2024/1689, artículo 9 — ¿Qué medidas de gestión de riesgos se exigen para los sistemas de IA de alto riesgo?
  [0.882] Reglamento (UE) 2024/1689 — Preámbulo y disposiciones iniciales — data/Reglamento-Europeo-IA.md
  [0.856] Reglamento (UE) 2024/1689 — Preámbulo y disposiciones iniciales — data/Reglamento-Europeo-IA.md
  [0.844] Reglamento (UE) 2024/1689 — Preámbulo y disposiciones iniciales — data/Reglamento-Europeo-IA.md
NO ENCONTRADO EN TOP 5: Reglamento (UE) 2016/679, artículo 15 — ¿Qué establece el derecho de acceso de los interesados a sus datos personales?
  [0.833] Reglamento (UE) 2016/679 — Preámbulo y disposiciones iniciales — data/L00001-00088.md
  [0.818] BOE-A-2021-8806 — 22 — data/BOE-A-2021-8806-consolidado.md
  [0.814] Reglamento (UE) 2016/679 — Preámbulo y disposiciones iniciales — data/L00001-00088.md
NO ENCONTRADO EN TOP 5: BOE-A-2018-16673, artículo 13 — ¿Cómo se regula el derecho de acceso a los datos personales en España?
  [0.78

## 9. Cómo reutilizar el índice

El índice queda guardado en `rag_store/`, una carpeta local excluida de Git. `rag.py` es la conexión compartida para cargarlo, recuperar fragmentos y consultar el modelo local mediante Ollama. Conserva los tres archivos: `index.faiss`, `chunks.json` y `metadata.json`.

```python
import rag

store = rag.load_rag_store()
for score, chunk in store.search("¿Qué establece el artículo 9 del Reglamento de IA?"):
    print(f"{rag.format_citation(chunk)} [{score:.3f}]")
```

Para consultar con respuestas generadas usa `python chat.py` o `python -m streamlit run app.py`. Ambos llaman a `rag.answer_question` con `qwen2.5:3b` en Ollama por defecto. La consulta requiere ejecutar el modelo de embeddings; la generación requiere además el LLM. No ejecutes estas consultas ni las celdas de embeddings en un equipo sin recursos suficientes.